In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

AGG_DIR = PROJECT_DIR / "data" / "processed" / "aggregations_v1"
FINAL_DIR = PROJECT_DIR / "data" / "processed" / "atlas_outputs_v1"
ARCHIVE_DIR = AGG_DIR / "_archive_legacy_outputs"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

K = 7

In [2]:
ward_path = AGG_DIR / "k7_ward25_named_full_profile_v1_enriched.csv"
ward = pd.read_csv(ward_path, low_memory=False)

required = ["LAD25CD", "LAD25NM", "WD25CD", "WD25NM"]

missing = [c for c in required if c not in ward.columns]

if missing:
    print("Missing columns:", missing)
    print("Use the lad_fixed file instead.")
    
    fallback_path = AGG_DIR / "k7_ward25_named_full_profile_v1_enriched_lad_fixed.csv"
    ward = pd.read_csv(fallback_path, low_memory=False)
    
    missing_fallback = [c for c in required if c not in ward.columns]
    if missing_fallback:
        raise ValueError(f"Fallback also missing columns: {missing_fallback}")
else:
    print("Ward enriched file has required LAD/Ward columns.")

Ward enriched file has required LAD/Ward columns.


In [3]:
canonical_ward_path = FINAL_DIR / "k7_ward25_atlas_profile_v1.csv"
ward.to_csv(canonical_ward_path, index=False)

print("Saved:", canonical_ward_path)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\atlas_outputs_v1\k7_ward25_atlas_profile_v1.csv


In [4]:
canonical_ward_path = FINAL_DIR / "k7_ward25_atlas_profile_v1.csv"
ward.to_csv(canonical_ward_path, index=False)

print("Saved:", canonical_ward_path)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\atlas_outputs_v1\k7_ward25_atlas_profile_v1.csv


In [5]:
manifest = pd.DataFrame([
    {
        "file": "k7_cluster_interpretation_key_v1.csv",
        "level": "cluster",
        "purpose": "Plain-English interpretation of each K=7 tribe.",
        "use_this_for": "Reports, legends, methodology notes.",
    },
    {
        "file": "k7_oa_geo_cluster_base_v1.csv",
        "level": "OA",
        "purpose": "Base OA-level geography and cluster assignment table.",
        "use_this_for": "Re-aggregation, debugging, future lookup joins.",
    },
    {
        "file": "k7_lsoa21_named_full_profile_v1.csv",
        "level": "LSOA21",
        "purpose": "LSOA-level cluster and demographic profile.",
        "use_this_for": "Neighbourhood-level analysis.",
    },
    {
        "file": "k7_msoa21_named_full_profile_v1.csv",
        "level": "MSOA21",
        "purpose": "MSOA-level cluster and demographic profile.",
        "use_this_for": "Medium-area mapping and interpretation.",
    },
    {
        "file": "k7_ward25_atlas_profile_v1.csv",
        "level": "Ward25",
        "purpose": "Canonical ward-level atlas profile with LAD fields.",
        "use_this_for": "North West ward analysis and target-readiness work.",
    },
    {
        "file": "k7_lad25_named_full_profile_v1.csv",
        "level": "LAD25",
        "purpose": "Council-level cluster and demographic profile.",
        "use_this_for": "Council comparison and regional summaries.",
    },
    {
        "file": "k7_ward25_map_ready_v1.csv",
        "level": "Ward25",
        "purpose": "Slim map-join file for ward boundary mapping.",
        "use_this_for": "QGIS / GeoPandas ward maps.",
    },
    {
        "file": "k7_msoa21_map_ready_v1.csv",
        "level": "MSOA21",
        "purpose": "Slim map-join file for MSOA boundary mapping.",
        "use_this_for": "QGIS / GeoPandas MSOA maps.",
    },
])

manifest.to_csv(FINAL_DIR / "atlas_outputs_manifest_v1.csv", index=False)
manifest

,file,level,purpose,use_this_for
0,k7_cluster_interpretation_key_v1.csv,cluster,Plain-English interpretation of each K=7 tribe.,"Reports, legends, methodology notes."
1,k7_oa_geo_cluster_base_v1.csv,OA,Base OA-level geography and cluster assignment...,"Re-aggregation, debugging, future lookup joins."
2,k7_lsoa21_named_full_profile_v1.csv,LSOA21,LSOA-level cluster and demographic profile.,Neighbourhood-level analysis.
3,k7_msoa21_named_full_profile_v1.csv,MSOA21,MSOA-level cluster and demographic profile.,Medium-area mapping and interpretation.
4,k7_ward25_atlas_profile_v1.csv,Ward25,Canonical ward-level atlas profile with LAD fi...,North West ward analysis and target-readiness ...
5,k7_lad25_named_full_profile_v1.csv,LAD25,Council-level cluster and demographic profile.,Council comparison and regional summaries.
6,k7_ward25_map_ready_v1.csv,Ward25,Slim map-join file for ward boundary mapping.,QGIS / GeoPandas ward maps.
7,k7_msoa21_map_ready_v1.csv,MSOA21,Slim map-join file for MSOA boundary mapping.,QGIS / GeoPandas MSOA maps.


# Deliverable: North West Tribe Atlas v1

1. K=7 tribe interpretation key
2. North West council composition table
3. Ward-level dominant tribe map
4. Ward-level fragmentation map
5. List of high-dominance wards by tribe
6. List of mixed/fragmented wards
7. Boundary/data caveats
8. Next required data: election results and membership/candidate capacity

---

1. Clean/canonicalise the output folder.
2. Create atlas_outputs_v1.
3. Produce North West summary/watchlist tables.
4. Make maps.
5. Build election-results ingestion.
6. Only then start target scoring.

In [6]:
ward = pd.read_csv(FINAL_DIR / "k7_ward25_atlas_profile_v1.csv", low_memory=False)

north_west_lads = [
    "Cheshire East", "Cheshire West and Chester", "Halton", "Warrington",
    "Cumberland", "Westmorland and Furness",
    "Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford",
    "Stockport", "Tameside", "Trafford", "Wigan",
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",
    "Knowsley", "Liverpool", "Sefton", "St. Helens", "Wirral",
]

nw = ward[ward["LAD25NM"].isin(north_west_lads)].copy()

nw.to_csv(FINAL_DIR / "k7_north_west_ward25_atlas_profile_v1.csv", index=False)

print("North West ward rows:", len(nw))
print("North West LADs:", nw["LAD25NM"].nunique())

North West ward rows: 825
North West LADs: 35


In [7]:
cluster_share_cols = [c for c in nw.columns if c.endswith("_share") and not c.startswith("dominant") and not c.startswith("second")]

lad_summary = (
    nw.groupby(["LAD25CD", "LAD25NM"], as_index=False)
    .agg(
        ward_count=("WD25CD", "nunique"),
        population=("population", "sum"),
        avg_dominant_cluster_share=("dominant_cluster_share", "mean"),
        avg_fragmentation=("cluster_fragmentation_index", "mean"),
    )
)

# Weighted council cluster shares from ward-level cluster populations if available
cluster_pop_cols = [c for c in nw.columns if c.startswith("cluster_") and c.endswith("_population")]

cluster_by_lad = (
    nw.groupby(["LAD25CD", "LAD25NM"], as_index=False)[cluster_pop_cols]
    .sum()
)

for c in cluster_pop_cols:
    share_col = c.replace("_population", "_share")
    cluster_by_lad[share_col] = cluster_by_lad[c] / cluster_by_lad[cluster_pop_cols].sum(axis=1)

lad_summary = lad_summary.merge(cluster_by_lad, on=["LAD25CD", "LAD25NM"], how="left")

lad_summary.to_csv(FINAL_DIR / "k7_north_west_lad25_cluster_summary_v1.csv", index=False)

In [8]:
high_dominance = nw[nw["dominant_cluster_share"] >= 0.60].copy()

high_dominance = high_dominance.sort_values(
    ["dominant_cluster_name", "dominant_cluster_share"],
    ascending=[True, False]
)

high_dominance.to_csv(FINAL_DIR / "k7_north_west_high_dominance_wards_v1.csv", index=False)

In [9]:
mixed = nw[nw["dominant_cluster_share"] < 0.40].copy()

mixed = mixed.sort_values(
    ["cluster_fragmentation_index", "dominant_cluster_share"],
    ascending=[False, True]
)

mixed.to_csv(FINAL_DIR / "k7_north_west_mixed_fragmented_wards_v1.csv", index=False)

In [10]:
interesting_clusters = [
    "Post-Industrial Estates / Deprived Working Communities",
    "Settled Working Families / Skilled Trades Suburbs",
    "Rooted Older Homeowners",
]

interesting_cols = [
    f"{name}_share"
    for name in interesting_clusters
    if f"{name}_share" in nw.columns
]

nw["initial_demographic_relevance_score"] = nw[interesting_cols].sum(axis=1)

demographic_relevance = nw.sort_values(
    "initial_demographic_relevance_score",
    ascending=False
)

demographic_relevance.to_csv(
    FINAL_DIR / "k7_north_west_demographic_relevance_watchlist_v1.csv",
    index=False
)

# Next: Start Mapping for the NW Atlas

MSOA map:
boundary file key = MSOA21CD
CSV key = MSOA21CD
use file = k7_msoa21_map_ready_v1.csv

Ward map:
boundary file key = WD25CD
CSV key = WD25CD
use file = k7_ward25_map_ready_v1.csv

Maps to Produce:

1. Dominant cluster by ward
2. Dominant cluster by MSOA
3. Dominant cluster share / confidence
4. Fragmentation index
5. Individual cluster-share heatmaps

The most useful individual heatmaps will be:

Post-Industrial Estates / Deprived Working Communities_share
Settled Working Families / Skilled Trades Suburbs_share
Rooted Older Homeowners_share
Student & Transient Youth_share
Cosmopolitan Young Professional Core_share

# Then: Political / Electoral Results - Candidate level

create: local_election_results_raw_v1.csv

CANDIDATE level NOT WARD level

columns incl.:

election_date
election_year
council_name
lad_code
ward_name
ward_code
boundary_year
candidate_name
party_label
votes
elected
position
seats_available
electorate
turnout
valid_votes
source_url
source_notes

# THEN LATER: Ward Results

create: ward_result_summary_v1.csv

winning_party
runner_up_party
winning_votes
runner_up_votes
margin_votes
margin_pct
party_vote_shares
fragmentation
turnout